## Imports

In [ ]:
from pathlib import Path

import tiktoken

## Raw text

In [ ]:
TEXT_PATH = Path("..") / "the-verdict.txt"

raw_text = TEXT_PATH.read_text(encoding="utf-8")

print("Total number of characters:", len(raw_text))
print(raw_text[:99])

## Byte pair encoding

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = tokenizer.encode(raw_text)

print("Vocabulary size:", tokenizer.n_vocab)
print("Total number of tokens:", len(token_ids))
print(token_ids[:20])

## Round trip

In [ ]:
sample = "Hello, do you like tea?"

sample_ids = tokenizer.encode(sample)

print(sample_ids)
print(tokenizer.decode(sample_ids))

## Unknown words

In [ ]:
unknown_ids = tokenizer.encode("Akwirw ier")

# BPE never needs an <|unk|> token. A word it has not seen falls apart into
# subwords it has, and decoding those back reassembles the original exactly.
for token_id in unknown_ids:
    print(token_id, repr(tokenizer.decode([token_id])))

print(tokenizer.decode(unknown_ids))

## Special tokens

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
)

# <|endoftext|> is refused unless it is explicitly allowed, so that text
# arriving from elsewhere cannot smuggle a document boundary into the stream.
special_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(special_ids)
print(tokenizer.decode(special_ids))

## Input-target pairs

In [ ]:
CONTEXT_SIZE = 4

# Skip ahead a little; the opening tokens make for a dull example.
window = token_ids[50:]

x = window[:CONTEXT_SIZE]
y = window[1 : CONTEXT_SIZE + 1]

# The target is just the input shifted one token left: at every position the
# model predicts the next token, so one slice of text yields CONTEXT_SIZE
# training examples rather than one.
print(f"x: {x}")
print(f"y: {y}")

In [ ]:
for i in range(1, CONTEXT_SIZE + 1):
    context = window[:i]
    desired = window[i]
    print(context, "--->", desired)

In [ ]:
for i in range(1, CONTEXT_SIZE + 1):
    context = window[:i]
    desired = window[i]
    print(tokenizer.decode(context), "--->", tokenizer.decode([desired]))